# Žmogaus valdymo ciklas su Microsoft Agent Framework

## 🎯 Mokymosi tikslai

Šiame užraše sužinosite, kaip įgyvendinti **žmogaus valdomus** darbo srautus naudojant Microsoft Agent Framework `RequestInfoExecutor`. Šis galingas modelis leidžia sustabdyti dirbtinio intelekto darbo srautus, kad būtų renkama žmogaus įvestis, taip padarant agentus interaktyviais ir suteikiant žmonėms kontrolę svarbiems sprendimams priimti.

## 🔄 Kas yra žmogaus valdymo ciklas?

**Žmogaus valdymo ciklas (HITL)** yra dizaino modelis, kai DI agentai stabdo veikimą ir prašo žmogaus įvesties prieš tęsdami. Tai būtina:

- ✅ **Kritiniams sprendimams** - Gauti žmogaus patvirtinimą prieš imantis svarbių veiksmų
- ✅ **Neaiškiose situacijose** - Leisti žmonėms paaiškinti, kai DI nesupranta
- ✅ **Vartotojų pageidavimams** - Klausti vartotojų pasirinkimų tarp kelių variantų
- ✅ **Atitikčiai ir saugumui** - Užtikrinti žmogaus priežiūrą reguliuojamoms operacijoms
- ✅ **Interaktyviai patirčiai** - Kurti pokalbių agentus, reaguojančius į vartotojo įvestį

## 🏗️ Kaip tai veikia Microsoft Agent Framework

Karkasas suteikia tris pagrindinius HITL komponentus:

1. **`RequestInfoExecutor`** - specialus vykdytojas, kuris sustabdo darbo srautą ir išmeta `RequestInfoEvent`
2. **`RequestInfoMessage`** - bazinė klasė tipizuotoms užklausų apkrovoms, siunčiamoms žmonėms
3. **`RequestResponse`** - sieja žmogaus atsakymus su originaliomis užklausomis naudodamas `request_id`

**Darbo srauto modelis:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Mūsų pavyzdys: Viešbučio rezervavimas su vartotojo patvirtinimu

Plėsime sąlyginį darbo srautą pridėdami žmogaus patvirtinimą **prieš** siūlant alternatyvias vietas:

1. Vartotojas prašo tikslinės vietos (pvz., "Paryžius")
2. `availability_agent` tikrina, ar yra kambarių
3. **Jei nėra kambarių** → `confirmation_agent` klausia "Ar norėtumėte matyti alternatyvas?"
4. Darbo srautas **sustoja** naudodamas `RequestInfoExecutor`
5. **Žmogus atsako** "taip" arba "ne" per konsolės įvestį
6. `decision_manager` nukreipia pagal atsakymą:
   - **Taip** → Rodyti alternatyvias vietas
   - **Ne** → Atšaukti rezervacijos užklausą
7. Pateikti galutinį rezultatą

Tai parodo, kaip suteikti vartotojams kontrolę agento pasiūlymams!

---

Pradėkime! 🚀


## 1 žingsnis: Importuoti reikalingas bibliotekas

Importuojame standartines Agentų sistemos komponentus bei **žmogaus įsitraukimą reikalaujančias klases**:
- `RequestInfoExecutor` - Vykdytojas, kuris sustabdo darbo eigą žmogaus įvesties laukimui
- `RequestInfoEvent` - Įvykis, iššaukiamas kai prašoma žmogaus įvesties
- `RequestInfoMessage` - Pagrindinė klasė tipizuotoms užklausų žinutėms
- `RequestResponse` - Susieja žmogaus atsakymus su užklausomis
- `WorkflowOutputEvent` - Įvykis darbo eigos rezultatų aptikimui


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## 2 žingsnis: Apibrėžkite Pydantic modelius struktūrizuotiems rezultatams

Šie modeliai apibrėžia **schemą**, kurią agentai grąžins. Mes išlaikome visus modelius iš sąlyginio darbo eigos ir pridedame:

**Nauja Žmogiškojo Kontrolės Etapo metu:**
- `HumanFeedbackRequest` - `RequestInfoMessage` potipis, apibrėžiantis užklausos duomenis, siunčiamus žmonėms
  - Apima `prompt` (klausimas, kurį užduoti) ir `destination` (kontekstas apie nepasiekiamą miestą)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## 3 žingsnis: Sukurkite viešbučių užsakymo įrankį

Tas pats įrankis iš sąlyginio darbo eigos – tikrina, ar paskirties vietoje yra laisvų kambarių.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## 4 žingsnis: Apibrėžkite sąlyginius routing'o funkcijas

Mums reikia **keturių sąlyginių funkcijų** mūsų žmogaus valdomam darbo eigai:

**Iš sąlyginės darbo eigos:**
1. `has_availability_condition` - Rutinimas, kai viešbučiai YRA prieinami
2. `no_availability_condition` - Rutinimas, kai viešbučiai NĖRA prieinami

**Nauja žmogaus įtraukimui:**
3. `user_wants_alternatives_condition` - Rutinimas, kai vartotojas sako „taip“ alternatyvoms
4. `user_declines_alternatives_condition` - Rutinimas, kai vartotojas sako „ne“ alternatyvoms


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## 5 žingsnis: Sukurkite Sprendimų Valdytojo Vykdytoją

Tai yra **žmogaus dalyvavimo ciklo modelio branduolys**! `DecisionManager` yra specialus `Executor`, kuris:

1. **Gaukia žmogaus atsiliepimus** per `RequestResponse` objektus
2. **Apdoroja vartotojo sprendimą** (taip/ne)
3. **Valdo darbo eigą** siųsdamas žinutes tinkamiems agentams

Pagrindinės savybės:
- Naudoja `@handler` dekoratorių metodams eksponuoti kaip darbo eigos žingsnius
- Priima `RequestResponse[HumanFeedbackRequest, str]`, kuriame yra tiek originalus užklausimas, tiek vartotojo atsakymas
- Grąžina paprastas "taip" arba "ne" žinutes, kurios suaktyvina mūsų sąlygos funkcijas


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## 6 Žingsnis: Sukurkite žodyninį vykdytoją

Tas pats vykdytojas kaip ir sąlyginio darbo eigos - pateikia galutinius rezultatus kaip darbo eigos išvestį.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## 7 žingsnis: Įkelkite aplinkos kintamuosius

Konfigūruokite LLM klientą (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## 8 žingsnis: Sukurkite DI agentus ir vykdytojus

Mes kuriame **šešis darbo eigos komponentus**:

**Agentai (įvynioti į AgentExecutor):**
1. **availability_agent** - Tikrina viešbučių prieinamumą naudojant įrankį
2. **confirmation_agent** - 🆕 Paruošia žmogaus patvirtinimo užklausą
3. **alternative_agent** - Siūlo alternatyvius miestus (kai vartotojas sako taip)
4. **booking_agent** - Skatina užsakymą (kai yra laisvų kambarių)
5. **cancellation_agent** - 🆕 Tvarko atšaukimo pranešimą (kai vartotojas sako ne)

**Specialūs vykdytojai:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, kuris sustabdo darbo eigą žmogaus įvesties laukimui
7. **decision_manager** - 🆕 Vartotojų atsakymų pagrindu maršrutuojantis pasirinktinis vykdytojas (jau apibrėžtas aukščiau)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## 9 žingsnis: Sukurkite darbo eigą su žmogumi cikle

Dabar sudarome darbo eigos grafiką su **sąlyginiu maršruto parinkimu**, įskaitant žmogaus ciklą:

**Darbo eigos struktūra:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Pagrindiniai kraštai:**
- `availability_agent → confirmation_agent` (kai nėra kambarių)
- `confirmation_agent → prepare_human_request` (transformacijos tipas)
- `prepare_human_request → request_info_executor` (pauzė žmogui)
- `request_info_executor → decision_manager` (visada - pateikia RequestResponse)
- `decision_manager → alternative_agent` (kai vartotojas sako „taip“)
- `decision_manager → cancellation_agent` (kai vartotojas sako „ne“)
- `availability_agent → booking_agent` (kai kambariai yra)
- Visi keliai baigiasi ties `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## 10 žingsnis: Vykdykite 1 testą - Miestas BE prieinamumo (Paryžius su žmogaus patvirtinimu)

Šis testas demonstruoja **pilną žmogaus ciklą**:

1. Užklausti viešbutį Paryžiuje
2. availability_agent tikrina → Nėra kambarių
3. confirmation_agent sukuria su žmogumi susijusį klausimą
4. request_info_executor **sustabdo darbo eigą** ir išveda `RequestInfoEvent`
5. **Programa aptinka įvykį ir paskatina vartotoją konsolėje**
6. Vartotojas įveda "taip" arba "ne"
7. Programa siunčia atsakymą per `send_responses_streaming()`
8. decision_manager nukreipia pagal atsakymą
9. Pateikiama galutinė reikšmė

**Pagrindinis modelis:**
- Pirmam kartui naudokite `workflow.run_stream()`
- Vėlesnėms kartoms naudokite `workflow.send_responses_streaming(pending_responses)`
- Klausykite `RequestInfoEvent`, kad aptiktumėte, kada reikalingas žmogaus įvestis
- Klausykite `WorkflowOutputEvent`, kad gautumėte galutinius rezultatus


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## 11 veiksmas: vykdykite 2 testą – miestas SU prieinamumu (Stokholmas – nereikia žmogaus įvesties)

Šis testas demonstruoja **tiesioginį kelią**, kai kambariai yra laisvi:

1. Pateikti užklausą viešbučiui Stokholme
2. availability_agent tikrina → Kambariai laisvi ✅
3. booking_agent siūlo rezervaciją
4. display_result rodo patvirtinimą
5. **Nereikia žmogaus įvesties!**

Darbo eiga visiškai praleidžia žmogaus įtrauktą kelią, kai kambariai yra laisvi.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Pagrindinės išvados ir geriausios praktikos su žmogumi procese

### ✅ Ko išmokote:

#### 1. **RequestInfoExecutor šablonas**
Žmogaus procese šablonas Microsoft Agent Framework naudoja tris pagrindines sudedamąsias dalis:
- `RequestInfoExecutor` - Sustabdo darbų eigą ir siunčia įvykius
- `RequestInfoMessage` - Bazinė klasė tipizuotoms užklausų apvalkalams (kurkite jos subklases!)
- `RequestResponse` - Susieja žmogaus atsakymus su originaliomis užklausomis

**Svarbus supratimas:**
- `RequestInfoExecutor` pats nepriima įvesties – jis tik sustabdo darbų eigą
- Jūsų programos kodas turi klausytis `RequestInfoEvent` ir surinkti įvestį
- Turite kviesti `send_responses_streaming()` su žodynu, kuriame `request_id` atitinka vartotojo atsakymą

#### 2. **Transliacijos vykdymo šablonas**
```python
# Pirmas iteravimas
stream = workflow.run_stream(initial_request)

# Vėlesni iteravimai (po žmogaus įvesties)
stream = workflow.send_responses_streaming(pending_responses)

# Visada apdoroti įvykius
events = [event async for event in stream]
```

#### 3. **Įvykių varoma architektūra**
Klausykite konkrečių įvykių, kad valdytumėte darbų eigą:
- `RequestInfoEvent` - Reikalinga žmogaus įvestis (darbų eiga sustabdyta)
- `WorkflowOutputEvent` - Galutinis rezultatas prieinamas (darbų eiga užbaigta)
- `WorkflowStatusEvent` - Būsenų pokyčiai (VYKDOMA, LAUKIAMA SU UŽKLAUSOMIS ir kt.)

#### 4. **Individualūs vykdytojai su @handler**
`DecisionManager` parodo, kaip kurti vykdytojus, kurie:
- Naudoja `@handler` dekoratorių metodams atskleisti kaip darbų eigos žingsnius
- Gauti tipizuotus pranešimus (pvz., `RequestResponse[HumanFeedbackRequest, str]`)
- Vykdyti darbo eigą siųsdami pranešimus kitiems vykdytojams
- Pasiekti kontekstą per `WorkflowContext`

#### 5. **Sąlyginis maršrutavimas su žmogaus sprendimais**
Galite kurti sąlygos funkcijas, vertinančias žmogaus atsakymus:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Realūs panaudojimo atvejai:

1. **Patvirtinimo darbų eiga**
   - Gauti vadovo patvirtinimą prieš apdorojant išlaidų ataskaitas
   - Reikalauti žmogaus peržiūros prieš siunčiant automatinius el. laiškus
   - Patvirtinti didelės vertės sandorius prieš vykdymą

2. **Turinio moderavimas**
   - Pažymėti abejotiną turinį žmogaus peržiūrai
   - Paprašyti moderatorių priimti galutinį sprendimą ribiniais atvejais
   - Siųsti žmogui, kai DI pasitikėjimas yra žemas

3. **Klientų aptarnavimas**
   - Leisti DI automatiškai tvarkyti rutinius klausimus
   - Perduoti sudėtingus klausimus žmogaus agentams
   - Paklausti kliento, ar nori kalbėtis su žmogumi

4. **Duomenų apdorojimas**
   - Paprašyti žmonių išspręsti dvikryptiškus duomenų įrašus
   - Patvirtinti DI interpretacijas neaiškiems dokumentams
   - Leisti vartotojams rinktis tarp kelių tinkamų interpretacijų

5. **Saugumo kritinės sistemos**
   - Reikalauti žmogaus patvirtinimo prieš nepakeičiamus veiksmus
   - Gauti patvirtinimą prieš prieigą prie jautrių duomenų
   - Patvirtinti sprendimus reguliuojamose srityse (sveikata, finansai)

6. **Interaktyvūs agentai**
   - Kurti pokalbių robotus, kurie užduoda papildomus klausimus
   - Kurti vedlius, kurie nukreipia vartotojus per sudėtingus procesus
   - Projektuoti agentus, kurie žingsnis po žingsnio bendradarbiauja su žmonėmis

### 🔄 Palyginimas: su žmogumi procese ir be jo

| Funkcija | Sąlyginė darbų eiga | Žmogaus procese darbų eiga |
|---------|---------------------|---------------------------|
| **Vykdymas** | Vienas `workflow.run()` | Ciklas su `run_stream()` + `send_responses_streaming()` |
| **Vartotojo įvestis** | Nėra (visiškai automatizuota) | Interaktyvūs užklausimai per `input()` arba UI |
| **Sudedamosios dalys** | Agentai + Vykdytojai | + RequestInfoExecutor + DecisionManager |
| **Įvykiai** | Tik AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent ir kt. |
| **Sustabdymas** | Nesustabdo | Darbų eiga sustabdo RequestInfoExecutor |
| **Žmogaus kontrolė** | Nėra žmogaus kontrolės | Žmonės priima pagrindinius sprendimus |
| **Panaudojimo atvejis** | Automatizavimas | Bendradarbiavimas ir priežiūra |

### 🚀 Pažangūs šablonai:

#### Keli žmogaus sprendimų taškai
Gali būti keli `RequestInfoExecutor` mazgai toje pačioje darbų eigoje:
```python
.add_edge(agent1, request_info_1)  # Pirmas žmogaus sprendimas
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Antras žmogaus sprendimas
.add_edge(decision_manager_2, final_agent)
```

#### Laiko limitų valdymas
Įgyvendinkite laiko limitus žmogaus atsakymams:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Pagal numatytuosius nustatymus pasirinkite saugią parinktį
```

#### Turtingos vartotojo sąsajos integravimas
Vietoje `input()`, integruokite su žiniatinklio UI, Slack, Teams ir kt.:
```python
if isinstance(event, RequestInfoEvent):
    # Siųsti pranešimą vartotojo pageidaujamu kanalu
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Sąlyginis žmogus procese
Žmogiškos įvesties prašykite tik tam tikrose situacijose:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Siųsti tik žmogui, jei pasitikėjimas yra mažas arba reikšmė yra didelė
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Geriausios praktikos:

1. **Visada kurkite RequestInfoMessage subklases**
   - Užtikrina tipų saugumą ir validaciją
   - Leidžia sukurti turtingą kontekstą vartotojo sąsajai
   - Aiškiai parodo kiekvieno užklausos tipo paskirtį

2. **Naudokite aprašomuosius užklausimus**
   - Pateikite kontekstą apie tai, ko prašote
   - Paaiškinkite kiekvieno pasirinkimo pasekmes
   - Užduokite paprastus ir aiškius klausimus

3. **Tvarkykite netikėtą įvestį**
   - Validuokite vartotojo atsakymus
   - Pateikite numatytąsias reikšmes neteisingai įvedus
   - Suteikite aiškius klaidų pranešimus

4. **Sekite užklausų ID**
   - Naudokite koreliaciją tarp request_id ir atsakymų
   - Nesistenkite rankiniu būdu valdyti būsena

5. **Kurkite užblokuojančioms operacijoms netrikdant**
   - Nesustabdykite procesų laukdami įvesties
   - Naudokite asinchroninius šablonus visur
   - Remkite lygiagrečias darbų eigos egzistencijas

### 📚 Susijusios sąvokos:

- **Agentų tarpinė programinė įranga** - Perskirti agentų kvietimus (ankstesnis užrašas)
- **Darbų eigos būsenos valdymas** - Išsaugoti darbų eigos būseną tarp vykdymų
- **Daugiagentinė bendradarbiavimas** - Kombinuoti žmogaus procese su agentų komandomis
- **Įvykių varomos architektūros** - Kurti reaguojančias sistemas su įvykiais

---

### 🎓 Sveikiname!

Jūs įvaldėte žmogaus procese darbų eigos su Microsoft Agent Framework! Dabar žinote, kaip:
- ✅ Sustabdyti darbų eigą žmogaus įvesties surinkimui
- ✅ Naudoti RequestInfoExecutor ir RequestInfoMessage
- ✅ Tvarkyti transliacijos vykdymą su įvykiais
- ✅ Kurti individualius vykdytojus su @handler
- ✅ Maršrutuoti darbų eigą pagal žmogaus sprendimus
- ✅ Kurti interaktyvius DI agentus, bendradarbiaujančius su žmonėmis

**Tai yra svarbus šablonas kuriant patikimas, valdžias AI sistemas!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
